## Exercise 1

**Dataset Used:** Custom/Synthetic Array Data (numpy)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 10: Transfer Learning
# Niveau: Anfänger
# Aufgabe 1 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import tensorflow as tf
import numpy as np
import matplotlib

import matplotlib.pyplot as plt

print("TensorFlow Version:", tf.__version__)

# ── 1. Synthetischen 2-Klassen Datensatz erstellen ───────────────────────────
# Klasse 0: rote Bilder, Klasse 1: blaue Bilder (einfaches Beispiel)
np.random.seed(42)
BILDGROESSE = 96
N_KLASSEN   = 2
N_TRAIN     = 400
N_TEST      = 100

def synthetische_bilder_erstellen(n, bildgroesse, n_klassen):
    """Generiert farbige Bilder mit zufälligem Hintergrundrauschen."""
    X = np.random.rand(n, bildgroesse, bildgroesse, 3).astype("float32") * 0.3
    y = np.random.randint(0, n_klassen, size=n)
    for i in range(n):
        klasse = y[i]
        # Klasse 0: roter Kanal dominiert, Klasse 1: blauer Kanal dominiert
        if klasse == 0:
            X[i, :, :, 0] += 0.6   # Rot
        else:
            X[i, :, :, 2] += 0.6   # Blau
    X = np.clip(X, 0, 1)
    return X, y

X_train, y_train = synthetische_bilder_erstellen(N_TRAIN, BILDGROESSE, N_KLASSEN)
X_test,  y_test  = synthetische_bilder_erstellen(N_TEST,  BILDGROESSE, N_KLASSEN)

print(f"Trainingsdaten: {X_train.shape}  Labels: {y_train.shape}")

# ── 2. MobileNetV2 als Feature-Extractor laden ────────────────────────────────
# include_top=False: keine ImageNet-Klassifikationsschicht
# weights='imagenet': vortrainierte Gewichte aus ImageNet-Training
print("\nLade MobileNetV2 (ImageNet-Gewichte)...")
basismodell = tf.keras.applications.MobileNetV2(
    input_shape=(BILDGROESSE, BILDGROESSE, 3),
    include_top=False,
    weights="imagenet"
)

# Alle Schichten einfrieren (Gewichte werden NICHT trainiert)
basismodell.trainable = False

print(f"MobileNetV2 geladen: {basismodell.count_params():,} Parameter (eingefroren)")

# ── 3. Eigene Klassifikationsschicht hinzufügen ───────────────────────────────
modell = tf.keras.Sequential([
    basismodell,
    # Globales Average Pooling: von Feature Maps zu einem Vektor
    tf.keras.layers.GlobalAveragePooling2D(name="global_pooling"),
    tf.keras.layers.Dense(N_KLASSEN, activation="softmax", name="klassifikation"),
], name="MobileNetV2_Transfer")

modell.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
modell.summary()

print(f"\nTrainierbare Parameter: {modell.count_params() - basismodell.count_params():,}")
print(f"Eingefrorene Parameter: {basismodell.count_params():,}")

# ── 4. Training (nur Top-Schichten) ──────────────────────────────────────────
print("\nTrainiere nur die Top-Schichten...")
history = modell.fit(
    X_train, y_train,
    epochs=10,
    batch_size=16,
    validation_split=0.2,
    verbose=1
)

# ── 5. Evaluation ─────────────────────────────────────────────────────────────
test_loss, test_acc = modell.evaluate(X_test, y_test, verbose=0)
print(f"\nTest-Verlust:     {test_loss:.4f}")
print(f"Test-Genauigkeit: {test_acc:.4f}")

# ── 6. Konzept-Erklärung ausgeben ─────────────────────────────────────────────
print("\n── Was ist Transfer Learning? ──")
print("1. MobileNetV2 wurde auf 1.000.000+ ImageNet-Bilder trainiert.")
print("2. Es hat dabei Hierarchische Features gelernt (Kanten → Texturen → Objekte).")
print("3. Wir frieren diese Features ein (trainable=False).")
print("4. Nur die letzte Klassifikationsschicht wird auf unsere 2 Klassen trainiert.")
print("5. Vorteil: Wenig Daten und kurze Trainingszeit genügen!")

# ── 7. Visualisierung ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(13, 8))

# Beispielbilder
for i in range(3):
    axes[0, i].imshow(X_train[i])
    axes[0, i].set_title(f"Klasse: {'Rot' if y_train[i]==0 else 'Blau'}")
    axes[0, i].axis("off")

# Trainingsverlauf
axes[1, 0].plot(history.history["loss"],     label="Training-Loss")
axes[1, 0].plot(history.history["val_loss"], label="Validierungs-Loss")
axes[1, 0].set_title("Verlust")
axes[1, 0].set_xlabel("Epoche")
axes[1, 0].legend()
axes[1, 0].grid(True)

axes[1, 1].plot(history.history["accuracy"],     label="Training")
axes[1, 1].plot(history.history["val_accuracy"], label="Validierung")
axes[1, 1].set_title("Genauigkeit")
axes[1, 1].set_xlabel("Epoche")
axes[1, 1].legend()
axes[1, 1].grid(True)

# Modellarchitektur (Text)
axes[1, 2].text(0.05, 0.9, "MobileNetV2 (eingefroren)", fontsize=11,
                transform=axes[1, 2].transAxes, color="gray",
                bbox=dict(boxstyle="round", facecolor="lightgray"))
axes[1, 2].text(0.05, 0.6, "↓  GlobalAveragePooling2D", fontsize=11,
                transform=axes[1, 2].transAxes)
axes[1, 2].text(0.05, 0.4, "↓  Dense(2, softmax)", fontsize=11,
                transform=axes[1, 2].transAxes,
                bbox=dict(boxstyle="round", facecolor="lightgreen"))
axes[1, 2].text(0.05, 0.1, f"Trainierbar: {N_KLASSEN*1280+N_KLASSEN:,} Params",
                transform=axes[1, 2].transAxes, fontsize=9, color="green")
axes[1, 2].set_title("Modellarchitektur")
axes[1, 2].axis("off")

plt.suptitle("Feature Extraction mit MobileNetV2 (Transfer Learning)", fontsize=13)
plt.tight_layout()
plt.savefig("A10_1_mobilenet_transfer.png", dpi=100)
plt.show()
print("Diagramm gespeichert: A10_1_mobilenet_transfer.png")


## Exercise 2

**Dataset Used:** Custom/Synthetic Array Data (numpy)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 10: Transfer Learning
# Niveau: Anfänger
# Aufgabe 2 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import tensorflow as tf
import numpy as np
import matplotlib

import matplotlib.pyplot as plt

print("TensorFlow Version:", tf.__version__)

# ── 1. Datensatz erstellen ────────────────────────────────────────────────────
np.random.seed(42)
BILDGROESSE = 96
N_KLASSEN   = 2
N_TRAIN     = 400
N_TEST      = 100

def synthetische_bilder_erstellen(n, bildgroesse, n_klassen):
    X = np.random.rand(n, bildgroesse, bildgroesse, 3).astype("float32") * 0.3
    y = np.random.randint(0, n_klassen, size=n)
    for i in range(n):
        if y[i] == 0:
            X[i, :, :, 0] += 0.6  # Rot
        else:
            X[i, :, :, 2] += 0.6  # Blau
    return np.clip(X, 0, 1), y

X_train, y_train = synthetische_bilder_erstellen(N_TRAIN, BILDGROESSE, N_KLASSEN)
X_test,  y_test  = synthetische_bilder_erstellen(N_TEST,  BILDGROESSE, N_KLASSEN)

# ── 2. Phase 1: Eingefrорenes Basismodell (Feature Extraction) ────────────────
print("Phase 1: Feature Extraction (alle MobileNetV2-Schichten eingefroren)...")
basismodell = tf.keras.applications.MobileNetV2(
    input_shape=(BILDGROESSE, BILDGROESSE, 3),
    include_top=False, weights="imagenet"
)
basismodell.trainable = False

modell = tf.keras.Sequential([
    basismodell,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(N_KLASSEN, activation="softmax"),
], name="Phase1_Eingefroren")

modell.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_phase1 = modell.fit(
    X_train, y_train, epochs=5, batch_size=16,
    validation_split=0.2, verbose=1
)

acc_phase1 = modell.evaluate(X_test, y_test, verbose=0)[1]
print(f"\nPhase 1 Test-Genauigkeit (eingefroren): {acc_phase1:.4f}")
print(f"Trainierbare Params Phase 1: {sum(np.prod(v.shape) for v in modell.trainable_variables):,}")

# ── 3. Phase 2: Fine-Tuning der letzten 20 Schichten ─────────────────────────
print("\nPhase 2: Fine-Tuning (letzte 20 Schichten von MobileNetV2 auftauen)...")

# Basismodell auftauen
basismodell.trainable = True

# Alle Schichten außer den letzten 20 wieder einfrieren
for schicht in basismodell.layers[:-20]:
    schicht.trainable = False

# Aktive / Eingefrorene Schichten ausgeben
eingef = sum(1 for l in basismodell.layers if not l.trainable)
aufget = sum(1 for l in basismodell.layers if l.trainable)
print(f"Eingefrorene Schichten: {eingef}  |  Aufgetaute Schichten: {aufget}")

# SEHR niedrige Lernrate für Fine-Tuning (wichtig!)
modell.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print(f"Trainierbare Params Phase 2: {sum(np.prod(v.shape) for v in modell.trainable_variables):,}")

history_phase2 = modell.fit(
    X_train, y_train, epochs=10, batch_size=16,
    validation_split=0.2,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5,
                                          restore_best_weights=True, verbose=1)
    ],
    verbose=1
)

acc_phase2 = modell.evaluate(X_test, y_test, verbose=0)[1]
print(f"\nPhase 2 Test-Genauigkeit (Fine-Tuning): {acc_phase2:.4f}")
print(f"Verbesserung durch Fine-Tuning: {(acc_phase2 - acc_phase1)*100:+.2f}%")

# ── 4. Visualisierung ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Verlauf beider Phasen zusammenführen
val_accs_1 = history_phase1.history["val_accuracy"]
val_accs_2 = history_phase2.history["val_accuracy"]
ep_1 = range(1, len(val_accs_1) + 1)
ep_2 = range(len(val_accs_1) + 1, len(val_accs_1) + len(val_accs_2) + 1)

axes[0].plot(ep_1, val_accs_1, color="#3498db", linewidth=2,
             label="Phase 1: Feature Extraction")
axes[0].plot(ep_2, val_accs_2, color="#e74c3c", linewidth=2,
             label="Phase 2: Fine-Tuning")
axes[0].axvline(x=len(val_accs_1) + 0.5, color="black",
                linestyle="--", label="Fine-Tuning beginnt")
axes[0].set_title("Validierungs-Genauigkeit: Beide Phasen")
axes[0].set_xlabel("Epoche")
axes[0].set_ylabel("Genauigkeit")
axes[0].legend()
axes[0].grid(True)

# Vergleich
axes[1].bar(["Phase 1\n(Eingefroren)", "Phase 2\n(Fine-Tuning)"],
            [acc_phase1, acc_phase2],
            color=["#3498db", "#e74c3c"], edgecolor="black")
axes[1].set_title("Test-Genauigkeit Vergleich")
axes[1].set_ylabel("Genauigkeit")
axes[1].set_ylim(0, 1.1)
for i, a in enumerate([acc_phase1, acc_phase2]):
    axes[1].text(i, a + 0.02, f"{a:.4f}", ha="center", fontsize=12)
axes[1].grid(True, axis="y", alpha=0.3)

plt.suptitle("Fine-Tuning mit MobileNetV2:\n"
             "Phase 1 (eingefroren) → Phase 2 (letzte 20 Schichten auftauen)",
             fontsize=12)
plt.tight_layout()
plt.savefig("A10_2_fine_tuning.png", dpi=100)
plt.show()
print("Diagramm gespeichert: A10_2_fine_tuning.png")


## Exercise 3

**Dataset Used:** Custom/Synthetic Array Data (numpy)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 10: Transfer Learning
# Niveau: Anfänger
# Aufgabe 3 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import tensorflow as tf
import numpy as np
import matplotlib

import matplotlib.pyplot as plt

print("TensorFlow Version:", tf.__version__)

# ── 1. MobileNetV2 laden und erste Conv-Schicht-Filter visualisieren ──────────
print("Lade MobileNetV2 (ImageNet-Gewichte)...")
basismodell = tf.keras.applications.MobileNetV2(
    input_shape=(96, 96, 3),
    include_top=False,
    weights="imagenet"
)
basismodell.trainable = False

print(f"Modell geladen: {basismodell.count_params():,} Parameter")
print(f"Anzahl Schichten: {len(basismodell.layers)}")

# ── 2. Erste Schicht finden und Filter extrahieren ────────────────────────────
erste_conv_schicht = None
for schicht in basismodell.layers:
    if isinstance(schicht, tf.keras.layers.Conv2D):
        erste_conv_schicht = schicht
        break

if erste_conv_schicht is None:
    # MobileNetV2 verwendet manchmal andere Schichttypen für erste Faltung
    for schicht in basismodell.layers:
        if "conv" in schicht.name.lower() and hasattr(schicht, "kernel"):
            erste_conv_schicht = schicht
            break

if erste_conv_schicht is not None:
    gewichte = erste_conv_schicht.get_weights()[0]  # (H, W, C_in, C_out)
    print(f"\nErste Conv-Schicht: {erste_conv_schicht.name}")
    print(f"Filter-Shape: {gewichte.shape}")
    # → (3, 3, 3, 32): 32 Filter, je 3×3 Pixel, 3 Eingabekanäle (RGB)
else:
    # Fallback: Zufällige Filter zur Illustration
    print("Fallback: Generiere Beispiel-Filter...")
    gewichte = np.random.randn(3, 3, 3, 32).astype("float32") * 0.1

# ── 3. Filter-Visualisierung ──────────────────────────────────────────────────
def filter_normalisieren(f):
    """Normalisiert einen Filter auf [0, 1] für die Anzeige."""
    f_min, f_max = f.min(), f.max()
    if f_max - f_min > 1e-6:
        return (f - f_min) / (f_max - f_min)
    return f - f_min

n_filter_zeigen = min(32, gewichte.shape[3])
n_spalten = 8
n_zeilen  = (n_filter_zeigen + n_spalten - 1) // n_spalten

fig, axes = plt.subplots(n_zeilen, n_spalten, figsize=(n_spalten * 2, n_zeilen * 2))
axes = axes.flatten()

for i in range(n_filter_zeigen):
    f = gewichte[:, :, :, i]   # (3, 3, 3) – ein RGB-Filter
    f_norm = filter_normalisieren(f)
    axes[i].imshow(f_norm)
    axes[i].set_title(f"F{i+1}", fontsize=7)
    axes[i].axis("off")

for i in range(n_filter_zeigen, len(axes)):
    axes[i].axis("off")

plt.suptitle(f"MobileNetV2: Gelernte Filter der ersten Conv2D-Schicht\n"
             f"(ImageNet-Gewichte, {n_filter_zeigen} von {gewichte.shape[3]} Filter gezeigt)",
             fontsize=12)
plt.tight_layout()
plt.savefig("A10_3_imagenet_filter.png", dpi=100)
plt.show()
print("Diagramm gespeichert: A10_3_imagenet_filter.png")

# ── 4. Feature-Hierarchie erklären ────────────────────────────────────────────
print("\n── Was wird in ImageNet-Modellen gelernt? ──")
print("Schicht 1–3 (früh):   Niedrige Features")
print("  → Kanten, Linien, einfache Texturen, Farbübergänge")
print("Schicht 4–8 (mittel): Mittlere Features")
print("  → Ecken, Bögen, Gitter, Lochstrukturen")
print("Schicht 9+ (tief):    Hohe Features")
print("  → Augen, Räder, Fell-Texturen, Gesichtsteile")
print()
print("Transfer Learning nutzt diese hierarchischen Features!")
print("→ Für neue Aufgaben reicht oft nur das Fine-Tuning der letzten Schichten.")

# ── 5. Schichtenstruktur von MobileNetV2 ausgeben ────────────────────────────
print(f"\n── MobileNetV2 Schichtenstruktur (Zusammenfassung) ──")
typen = {}
for schicht in basismodell.layers:
    typ = type(schicht).__name__
    typen[typ] = typen.get(typ, 0) + 1

print("Schicht-Typen:")
for typ, anzahl in sorted(typen.items(), key=lambda x: -x[1]):
    print(f"  {typ:<30}: {anzahl}×")

# ── 6. Aktivierungsvisualisierung einer echten ImageNet-Filterantwort ─────────
# Einfaches Test-Bild
test_bild = np.random.rand(1, 96, 96, 3).astype("float32")
test_bild[0, 30:60, 30:60, :] = 0.9  # weißes Quadrat

# Feature-Map extrahieren
fm_modell = tf.keras.Model(
    inputs=basismodell.input,
    outputs=erste_conv_schicht.output if erste_conv_schicht else basismodell.layers[1].output,
    name="FM_Extrator"
)
feature_maps = fm_modell.predict(test_bild, verbose=0)
print(f"\nFeature-Map-Shape nach Conv1: {feature_maps.shape}")

fig2, axes2 = plt.subplots(2, 8, figsize=(16, 4))
axes2[0, 0].imshow(test_bild[0])
axes2[0, 0].set_title("Eingabe")
axes2[0, 0].axis("off")
for j in range(1, 8):
    axes2[0, j].axis("off")

n_fm = min(16, feature_maps.shape[-1])
for j in range(n_fm):
    zeile = j // 8 if n_fm > 8 else 1
    spalte = j % 8
    fm = feature_maps[0, :, :, j]
    fm_norm = filter_normalisieren(fm)
    axes2[1, j].imshow(fm_norm, cmap="viridis")
    axes2[1, j].set_title(f"FM {j+1}", fontsize=7)
    axes2[1, j].axis("off")

plt.suptitle("Feature Maps der ersten Conv-Schicht für ein Testbild", fontsize=12)
plt.tight_layout()
plt.savefig("A10_3_feature_maps_eingabe.png", dpi=100)
plt.show()
print("Diagramm gespeichert: A10_3_feature_maps_eingabe.png")
